# Statistical Analysis of Drone and Bird Radar Signatures

This notebook quantitatively analyses real drone and bird measurements acquired with a 77 GHz SAAB SIRS 1600 FMCW radar.

The objectives are to:

1. Construct and validate a segment-level metadata index.
2. Examine class imbalance and target-range bias.
3. Extract interpretable Doppler features.
4. Compare broad target groups and original subtypes.
5. Repeat the analysis under range-controlled conditions.
6. Validate the findings at measurement-session level.
7. Establish methodological requirements for classification and synthetic augmentation.

## 1. Dataset Loading and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

FILE_PATH = Path("../data/raw/data_SAAB_SIRS_77GHz_FMCW.npy")

if not FILE_PATH.exists():
    raise FileNotFoundError(
        f"Dataset file not found: {FILE_PATH.resolve()}"
    )

data = np.load(FILE_PATH, allow_pickle=True)

print("Dataset shape:", data.shape)
print("Dataset dtype:", data.dtype)

The dataset has shape **(130, 6)** and NumPy object type. Each row represents one measurement session with a variable number of radar segments and associated metadata.

## 2. Target Definitions and Segment-Level Metadata

In [ ]:
DRONE_LABELS = {"D1", "D2", "D3", "D4", "D5", "D6"}

BIRD_LABELS = {
    "seagull",
    "pigeon",
    "raven",
    "black-headed gull",
    "seagull and black-headed gull",
    "heron"
}

PRF_HZ = 17_000
CARRIER_FREQUENCY_HZ = 77e9
SPEED_OF_LIGHT_M_S = 299_792_458

WAVELENGTH_M = SPEED_OF_LIGHT_M_S / CARRIER_FREQUENCY_HZ

AZIMUTH_START = 54
AZIMUTH_END = 204


def extract_label(value):
    label_array = np.asarray(value).reshape(-1)

    if label_array.size == 0:
        return "unknown"

    return str(label_array[0]).strip()


def map_target_group(label):
    if label in DRONE_LABELS:
        return "drone"

    if label in BIRD_LABELS:
        return "bird"

    return "excluded"

The six drone models D1–D6 are mapped to the **drone** group. All bird species and mixed-bird observations are mapped to **bird**. Humans and the corner reflector are excluded from the binary analysis.

In [ ]:
segment_records = []

for session_id, row in enumerate(data):
    label = extract_label(row[0])
    target_group = map_target_group(label)

    if target_group == "excluded":
        continue

    segments = np.asarray(row[1])
    ranges = np.asarray(row[2]).reshape(-1)
    times = np.asarray(row[3]).reshape(-1)
    splits = np.asarray(row[4]).reshape(-1)
    edge_flags = np.asarray(row[5]).reshape(-1)

    for segment_id in range(segments.shape[1]):
        segment_records.append({
            "session_id": session_id,
            "segment_id": segment_id,
            "label": label,
            "target_group": target_group,
            "range_m": float(ranges[segment_id]),
            "time_s": float(times[segment_id]),
            "split": int(splits[segment_id]),
            "edge_flag": int(edge_flags[segment_id])
        })

segments_df = pd.DataFrame(segment_records)

print("Selected drone and bird segments:", len(segments_df))
print("Field-of-view edge samples:", segments_df["edge_flag"].sum())

display(segments_df.head())

### Metadata-Index Result

The index contains **66,560 drone and bird segments**. Each record identifies the session, segment, original label, broad group, range, timestamp, official split, and field-of-view edge status. It references the original signal without duplicating the large complex matrices.

In [ ]:
valid_segments_df = (
    segments_df
    .query("edge_flag == 0")
    .reset_index(drop=True)
)

group_counts = (
    valid_segments_df
    .groupby("target_group")
    .size()
    .rename("segments")
    .to_frame()
)

print("Valid segments:", len(valid_segments_df))
display(group_counts)

### Valid Observation Counts

After excluding 67 field-of-view edge observations, **66,493 valid segments** remain:

| Target group | Valid segments |
|---|---:|
| Bird | 7,736 |
| Drone | 58,757 |

There are approximately **7.6 times more valid drone segments than bird segments**. Accuracy alone would therefore be misleading. Modelling will prioritise macro-F1, balanced accuracy, class recall, and confusion matrices.

## 3. Target-Range Distribution

In [ ]:
range_summary = (
    valid_segments_df
    .groupby("target_group")["range_m"]
    .agg(["count", "mean", "std", "median", "min", "max"])
)

display(range_summary)

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=valid_segments_df,
    x="range_m",
    hue="target_group",
    bins=50,
    stat="density",
    common_norm=False,
    element="step",
    fill=False
)

plt.xlabel("Target range (m)")
plt.ylabel("Density")
plt.title("Target Range Distribution: Drone vs Bird")
plt.show()

### Range-Distribution Results

| Target group | Mean | Standard deviation | Median | Minimum | Maximum |
|---|---:|---:|---:|---:|---:|
| Bird | 46.73 m | 15.83 m | 44.34 m | 11.32 m | 194.34 m |
| Drone | 39.47 m | 25.47 m | 32.08 m | 6.60 m | 145.28 m |

Bird observations have a median range approximately **12.26 m greater** than drone observations. Although the distributions overlap, range is a potential confounder because it can affect signal-to-noise ratio, spectral contrast, clutter, and visibility of weak Doppler components.

### 3.1 Statistical Range Comparison

In [ ]:
from scipy.stats import ks_2samp

drone_ranges = valid_segments_df.loc[
    valid_segments_df["target_group"] == "drone",
    "range_m"
].to_numpy()

bird_ranges = valid_segments_df.loc[
    valid_segments_df["target_group"] == "bird",
    "range_m"
].to_numpy()

ks_statistic, ks_p_value = ks_2samp(
    drone_ranges,
    bird_ranges
)

print("Drone median range (m):", np.median(drone_ranges))
print("Bird median range (m):", np.median(bird_ranges))
print("Drone mean range (m):", np.mean(drone_ranges))
print("Bird mean range (m):", np.mean(bird_ranges))
print("Kolmogorov–Smirnov statistic:", ks_statistic)
print("Kolmogorov–Smirnov p-value:", ks_p_value)

In [ ]:
print(f"Kolmogorov–Smirnov statistic: {ks_statistic:.4f}")
print(f"Kolmogorov–Smirnov p-value: {ks_p_value:.3e}")

The two-sample Kolmogorov–Smirnov test produced \(D=0.3624\), with a p-value below floating-point display precision. It is reported as \(p<0.001\), not \(p=0\).

The complete drone and bird range distributions differ substantially. Because consecutive segments from the same session are dependent, the p-value is interpreted cautiously; the KS statistic remains useful as a descriptive measure.

### 3.2 Shared-Range Candidate Interval

In [ ]:
OVERLAP_MIN_M = 30
OVERLAP_MAX_M = 75

range_matched_candidates = valid_segments_df.query(
    "@OVERLAP_MIN_M <= range_m <= @OVERLAP_MAX_M"
).copy()

range_overlap_summary = (
    range_matched_candidates
    .groupby("target_group")["range_m"]
    .agg(["count", "mean", "median", "min", "max"])
)

display(range_overlap_summary)

Restricting observations to **30–75 m** produces comparable global range statistics:

| Target group | Segments | Mean | Median | Minimum | Maximum |
|---|---:|---:|---:|---:|---:|
| Bird | 6,461 | 46.78 m | 45.28 m | 30.19 m | 74.53 m |
| Drone | 26,814 | 47.60 m | 46.23 m | 30.19 m | 74.53 m |

The mean and median differences are below one metre. This interval is suitable for a range-controlled experiment, although class balancing remains necessary.

## 4. Balanced Segment-Level Feature Extraction

In [ ]:
SAMPLES_PER_GROUP = 5_000
RANDOM_SEED = 42

feature_candidates = (
    valid_segments_df
    .groupby("target_group", group_keys=False)
    .sample(
        n=SAMPLES_PER_GROUP,
        random_state=RANDOM_SEED
    )
    .reset_index(drop=True)
)

display(
    feature_candidates
    .groupby("target_group")
    .size()
    .rename("samples")
    .to_frame()
)

A reproducible exploratory sample contains **5,000 valid segments from each target group**. Balancing prevents the larger drone class from dominating the descriptive comparison.

### 4.1 Range–Doppler Transformation

In [ ]:
# Radar configuration

PRF_HZ = 17_000
CARRIER_FREQUENCY_HZ = 77e9
SPEED_OF_LIGHT_M_S = 299_792_458

WAVELENGTH_M = SPEED_OF_LIGHT_M_S / CARRIER_FREQUENCY_HZ

AZIMUTH_START = 54
AZIMUTH_END = 204


def compute_range_doppler(segment):
    """
    Convert a complex radar segment of shape (5, 256)
    into a normalised range–Doppler representation
    of shape (5, 150).
    """
    segment = np.asarray(segment)

    if segment.shape != (5, 256):
        raise ValueError(
            f"Expected segment shape (5, 256), received {segment.shape}"
        )

    central_samples = segment[
        :,
        AZIMUTH_START:AZIMUTH_END
    ]

    window = np.hanning(central_samples.shape[1])

    windowed_signal = (
        central_samples * window[np.newaxis, :]
    )

    doppler_complex = np.fft.fftshift(
        np.fft.fft(windowed_signal, axis=1),
        axes=1
    )

    doppler_magnitude = np.abs(doppler_complex)

    doppler_db = 20 * np.log10(
        doppler_magnitude + 1e-12
    )

    doppler_db -= np.max(doppler_db)

    doppler_frequencies = np.fft.fftshift(
        np.fft.fftfreq(
            central_samples.shape[1],
            d=1 / PRF_HZ
        )
    )

    velocity_axis = (
        doppler_frequencies * WAVELENGTH_M / 2
    )

    return doppler_db, velocity_axis

Each complex segment is reshaped to **(5, 256)**. Central slow-time indices 54–203 are retained, a Hann window is applied, and an FFT is computed along slow time. The output is a normalised range–Doppler patch of shape **(5, 150)**.

This is a short observation covering approximately 15 ms, not a conventional long-duration micro-Doppler spectrogram.

### 4.2 Interpretable Doppler Features

In [ ]:
def extract_doppler_features(segment):
    """
    Extract interpretable features from one complex radar segment.
    """
    doppler_db, velocity = compute_range_doppler(segment)

    # Convert relative decibel values back to linear power.
    power = 10 ** (doppler_db / 10)

    # Total Doppler spectrum across all range cells.
    spectrum = power.sum(axis=0)
    total_energy = spectrum.sum() + 1e-12
    probability = spectrum / total_energy

    doppler_centroid = np.sum(velocity * probability)

    doppler_bandwidth = np.sqrt(
        np.sum(
            ((velocity - doppler_centroid) ** 2)
            * probability
        )
    )

    peak_velocity = velocity[np.argmax(spectrum)]

    spectral_entropy = -np.sum(
        probability * np.log(probability + 1e-12)
    ) / np.log(len(probability))

    near_zero_mask = np.abs(velocity) <= 1.0
    near_zero_energy_ratio = (
        spectrum[near_zero_mask].sum() / total_energy
    )

    high_velocity_mask = np.abs(velocity) >= 3.0
    high_velocity_energy_ratio = (
        spectrum[high_velocity_mask].sum() / total_energy
    )

    central_cell_energy_ratio = (
        power[2].sum() / (power.sum() + 1e-12)
    )

    return {
        "doppler_centroid_m_s": doppler_centroid,
        "doppler_bandwidth_m_s": doppler_bandwidth,
        "peak_velocity_m_s": peak_velocity,
        "spectral_entropy": spectral_entropy,
        "near_zero_energy_ratio": near_zero_energy_ratio,
        "high_velocity_energy_ratio": high_velocity_energy_ratio,
        "central_cell_energy_ratio": central_cell_energy_ratio
    }

Seven complementary features are extracted: Doppler centroid, bandwidth, peak velocity, entropy, near-zero energy ratio, high-velocity energy ratio, and central range-cell energy ratio.

In [ ]:
feature_records = []

for row in feature_candidates.itertuples(index=False):
    radar_segment = np.asarray(
        data[row.session_id, 1]
    )[:, row.segment_id].reshape(5, 256)

    features = extract_doppler_features(radar_segment)

    feature_records.append({
        "session_id": row.session_id,
        "segment_id": row.segment_id,
        "label": row.label,
        "target_group": row.target_group,
        "range_m": row.range_m,
        **features
    })

features_df = pd.DataFrame(feature_records)

print("Extracted feature rows:", len(features_df))
print(
    "Missing feature values:",
    features_df.isna().sum().sum()
)

display(features_df.head())

Feature extraction produced **10,000 rows with no missing values**, confirming numerical stability for the balanced exploratory sample.

In [ ]:
feature_columns = [
    "doppler_centroid_m_s",
    "doppler_bandwidth_m_s",
    "peak_velocity_m_s",
    "spectral_entropy",
    "near_zero_energy_ratio",
    "high_velocity_energy_ratio",
    "central_cell_energy_ratio"
]

feature_summary = (
    features_df
    .groupby("target_group")[feature_columns]
    .agg(["mean", "std", "median"])
)

display(feature_summary.round(4))

### Segment-Level Results

| Feature | Bird median | Drone median |
|---|---:|---:|
| Doppler centroid | −0.3535 m/s | −0.0677 m/s |
| Doppler bandwidth | 3.4087 m/s | 1.7550 m/s |
| Peak velocity | −0.4413 m/s | 0.0000 m/s |
| Spectral entropy | 0.4548 | 0.3483 |
| Near-zero energy ratio | 0.0314 | 0.8298 |
| High-velocity energy ratio | 0.5335 | 0.0495 |
| Central range-cell energy ratio | 0.5414 | 0.6060 |

Bird segments generally exhibit broader, more dispersed Doppler energy. Drone segments generally exhibit stronger near-zero concentration. Central range-cell energy differs less strongly.

## 5. Feature-Distribution Visualisation

In [ ]:
features_to_plot = [
    "doppler_bandwidth_m_s",
    "spectral_entropy",
    "near_zero_energy_ratio",
    "high_velocity_energy_ratio",
    "central_cell_energy_ratio"
]

plot_titles = {
    "doppler_bandwidth_m_s": "Doppler Bandwidth",
    "spectral_entropy": "Spectral Entropy",
    "near_zero_energy_ratio": "Near-Zero Energy Ratio",
    "high_velocity_energy_ratio": "High-Velocity Energy Ratio",
    "central_cell_energy_ratio": "Central Range-Cell Energy Ratio"
}

fig, axes = plt.subplots(
    2,
    3,
    figsize=(16, 9),
    constrained_layout=True
)

for axis, feature in zip(axes.flat, features_to_plot):
    sns.violinplot(
        data=features_df,
        x="target_group",
        y=feature,
        hue="target_group",
        order=["drone", "bird"],
        palette={"drone": "tab:blue", "bird": "tab:orange"},
        legend=False,
        inner="quartile",
        cut=0,
        ax=axis
    )

    axis.set_title(plot_titles[feature])
    axis.set_xlabel("Target group")
    axis.set_ylabel("Feature value")

axes.flat[-1].axis("off")

fig.suptitle(
    "Doppler Feature Distributions: Drone vs Bird",
    fontsize=16
)

plt.show()

### Distribution Interpretation

Bandwidth and entropy are generally higher for birds, but overlap is present. Near-zero and high-velocity ratios show the strongest differences and are visibly multimodal.

Most drone segments have high near-zero and low high-velocity energy, while most bird segments show the inverse. Long tails demonstrate that no single fixed threshold is reliable; a multivariate classifier is required. Central range-cell energy has the greatest overlap and is expected to provide supporting information.

## 6. Analysis by Original Target Subtype

In [ ]:
label_feature_summary = (
    features_df
    .groupby(["target_group", "label"])[
        [
            "doppler_bandwidth_m_s",
            "spectral_entropy",
            "near_zero_energy_ratio",
            "high_velocity_energy_ratio",
            "central_cell_energy_ratio"
        ]
    ]
    .median()
    .round(4)
)

display(label_feature_summary)

In [ ]:
features_by_label = [
    "doppler_bandwidth_m_s",
    "spectral_entropy",
    "near_zero_energy_ratio",
    "high_velocity_energy_ratio"
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(16, 10),
    constrained_layout=True
)

for axis, feature in zip(axes.flat, features_by_label):
    sns.boxplot(
        data=features_df,
        x="label",
        y=feature,
        hue="target_group",
        dodge=False,
        showfliers=False,
        ax=axis
    )

    axis.set_title(feature.replace("_", " ").title())
    axis.set_xlabel("Original target label")
    axis.tick_params(axis="x", rotation=45)

    if axis.get_legend() is not None:
        axis.get_legend().remove()

fig.suptitle(
    "Doppler Features by Drone Model and Bird Category",
    fontsize=16
)

plt.show()

### Subtype-Level Results

The broad classes are internally heterogeneous. D2, D3, and D6 generally have narrow bandwidth, low entropy, high near-zero energy, and low high-velocity energy. D1, D4, and D5 exhibit greater variability and overlap with birds; D5 has the greatest median bandwidth and entropy among drones.

Major gull categories and heron generally have broader, higher-entropy signatures. Pigeon and raven results are not independently reliable because the complete dataset contains only 32 and 19 segments, respectively.

Binary performance must therefore be accompanied by recall and confusion analysis for every original subtype.

## 7. Range-Controlled Feature Analysis

In [ ]:
RANGE_MIN_M = 30
RANGE_MAX_M = 75
SAMPLES_PER_GROUP = 5_000
RANDOM_SEED = 42

controlled_candidates = valid_segments_df.query(
    "@RANGE_MIN_M <= range_m <= @RANGE_MAX_M"
).copy()

controlled_sample = (
    controlled_candidates
    .groupby("target_group", group_keys=False)
    .sample(
        n=SAMPLES_PER_GROUP,
        random_state=RANDOM_SEED
    )
    .reset_index(drop=True)
)

controlled_sample_summary = (
    controlled_sample
    .groupby("target_group")["range_m"]
    .agg(["count", "mean", "std", "median", "min", "max"])
)

display(controlled_sample_summary.round(4))

The balanced controlled sample contains 5,000 observations per group between 30 m and 75 m. Mean ranges are 46.69 m for birds and 47.72 m for drones; medians are 45.28 m and 46.23 m.

In [ ]:
controlled_feature_records = []

for row in controlled_sample.itertuples(index=False):
    radar_segment = np.asarray(
        data[row.session_id, 1]
    )[:, row.segment_id].reshape(5, 256)

    extracted = extract_doppler_features(radar_segment)

    controlled_feature_records.append({
        "session_id": row.session_id,
        "segment_id": row.segment_id,
        "label": row.label,
        "target_group": row.target_group,
        "range_m": row.range_m,
        **extracted
    })

controlled_features_df = pd.DataFrame(
    controlled_feature_records
)

print(
    "Range-controlled feature rows:",
    len(controlled_features_df)
)
print(
    "Missing values:",
    controlled_features_df.isna().sum().sum()
)

The controlled extraction produced **10,000 feature rows with no missing values**.

In [ ]:
controlled_medians = (
    controlled_features_df
    .groupby("target_group")[feature_columns]
    .median()
    .T
)

display(controlled_medians.round(4))

### Range-Controlled Results

| Feature | Bird median | Drone median |
|---|---:|---:|
| Doppler centroid | −0.3124 m/s | −0.0625 m/s |
| Doppler bandwidth | 3.5017 m/s | 1.9414 m/s |
| Peak velocity | −0.4413 m/s | 0.0000 m/s |
| Spectral entropy | 0.4569 | 0.3645 |
| Near-zero energy ratio | 0.0316 | 0.7862 |
| High-velocity energy ratio | 0.4761 | 0.0692 |
| Central range-cell energy ratio | 0.5427 | 0.6033 |

The principal relationships persist after range control. Bird bandwidth remains approximately 1.80 times drone bandwidth. Range is therefore not the sole explanation for the observed separation.

## 8. Session-Level Validation

In [ ]:
session_feature_summary = (
    controlled_features_df
    .groupby(
        ["session_id", "target_group", "label"],
        as_index=False
    )[feature_columns]
    .median()
)

print(
    "Number of represented sessions:",
    session_feature_summary["session_id"].nunique()
)

display(
    session_feature_summary
    .groupby("target_group")
    .agg(
        sessions=("session_id", "nunique"),
        labels=("label", "nunique")
    )
)

In [ ]:
session_group_medians = (
    session_feature_summary
    .groupby("target_group")[feature_columns]
    .median()
    .T
)

display(session_group_medians.round(4))

The controlled sample represents **79 sessions**: 51 bird sessions across four represented labels and 28 drone sessions across all six models.

| Feature | Bird median | Drone median |
|---|---:|---:|
| Doppler centroid | −0.5637 m/s | −0.0311 m/s |
| Doppler bandwidth | 3.7293 m/s | 1.9258 m/s |
| Peak velocity | −0.6619 m/s | 0.1103 m/s |
| Spectral entropy | 0.4713 | 0.3761 |
| Near-zero energy ratio | 0.0278 | 0.3067 |
| High-velocity energy ratio | 0.5629 | 0.0676 |
| Central range-cell energy ratio | 0.5340 | 0.5996 |

In [ ]:
session_features_to_plot = [
    "doppler_bandwidth_m_s",
    "spectral_entropy",
    "near_zero_energy_ratio",
    "high_velocity_energy_ratio"
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 9),
    constrained_layout=True
)

for axis, feature in zip(
    axes.flat,
    session_features_to_plot
):
    sns.boxplot(
        data=session_feature_summary,
        x="target_group",
        y=feature,
        hue="target_group",
        order=["drone", "bird"],
        palette={
            "drone": "tab:blue",
            "bird": "tab:orange"
        },
        legend=False,
        showfliers=True,
        ax=axis
    )

    sns.stripplot(
        data=session_feature_summary,
        x="target_group",
        y=feature,
        order=["drone", "bird"],
        color="black",
        alpha=0.5,
        size=4,
        ax=axis
    )

    axis.set_title(
        feature.replace("_", " ").title()
    )
    axis.set_xlabel("Target group")

fig.suptitle(
    "Session-Level Doppler Feature Distributions",
    fontsize=16
)

plt.show()

### Session-Level Distribution Interpretation

Bandwidth provides relatively strong separation for most sessions, with two high drone outliers. Entropy remains higher for birds but overlaps more strongly.

Near-zero energy varies substantially among drone sessions, confirming subtype and flight-condition heterogeneity. Bird sessions remain mainly at low near-zero ratios. High-velocity energy shows the inverse pattern, with exceptions in both groups.

Persistence after session aggregation reduces the likelihood that long sessions alone produced the segment-level result.

## 9. Session-Level Statistical Testing

In [ ]:
from scipy.stats import mannwhitneyu

test_results = []

for feature in session_features_to_plot:
    drone_values = session_feature_summary.loc[
        session_feature_summary["target_group"] == "drone",
        feature
    ]

    bird_values = session_feature_summary.loc[
        session_feature_summary["target_group"] == "bird",
        feature
    ]

    statistic, p_value = mannwhitneyu(
        drone_values,
        bird_values,
        alternative="two-sided"
    )

    test_results.append({
        "feature": feature,
        "drone_sessions": len(drone_values),
        "bird_sessions": len(bird_values),
        "U_statistic": statistic,
        "p_value": p_value
    })

statistical_tests_df = pd.DataFrame(test_results)
display(statistical_tests_df.round(6))


Two-sided Mann–Whitney U tests compare drone and bird session medians. This non-parametric test is suitable for skewed distributions with outliers, using sessions rather than correlated segments as observational units.

In [ ]:
effect_size_results = []

for feature in session_features_to_plot:
    drone_values = session_feature_summary.loc[
        session_feature_summary["target_group"] == "drone",
        feature
    ]

    bird_values = session_feature_summary.loc[
        session_feature_summary["target_group"] == "bird",
        feature
    ]

    statistic, p_value = mannwhitneyu(
        drone_values,
        bird_values,
        alternative="two-sided"
    )

    n_drone = len(drone_values)
    n_bird = len(bird_values)

    rank_biserial = (
        2 * statistic / (n_drone * n_bird)
    ) - 1

    effect_size_results.append({
        "feature": feature,
        "U_statistic": statistic,
        "p_value": p_value,
        "rank_biserial_correlation": rank_biserial
    })

effect_sizes_df = pd.DataFrame(effect_size_results)

display(
    effect_sizes_df.style.format({
        "U_statistic": "{:.1f}",
        "p_value": "{:.3e}",
        "rank_biserial_correlation": "{:.4f}"
    })
)

### Statistical Significance and Effect Sizes

| Feature                       | U statistic | p-value                     | Rank-biserial correlation |
|-------------------------------|-------------:|-----------------------------:|---------------------------:|
| Doppler bandwidth              |        150.0 | $$7.681 \times 10^{-9}$$    | −0.7899                   |
| Spectral entropy               |        329.0 | $$8.122 \times 10^{-5}$$    | −0.5392                   |
| Near-zero energy ratio         |      1,132.0 | $$1.878 \times 10^{-5}$$    | +0.5854                   |
| High-velocity energy ratio     |        243.0 | $$1.420 \times 10^{-6}$$    | −0.6597                   |

For four comparisons, the Bonferroni-corrected threshold is

$$
\alpha_{\mathrm{corrected}} = \frac{0.05}{4} = 0.0125.
$$

All four differences remain significant. Doppler bandwidth has the largest absolute effect, followed by high-velocity energy, near-zero energy, and entropy. Negative effects indicate larger bird values; the positive near-zero effect indicates larger drone values.

## 10. Final Conclusion

This analysis establishes that drone and bird measurements differ in several interpretable Doppler characteristics.

The main findings are:

1. The valid dataset is strongly imbalanced: 58,757 drone versus 7,736 bird segments.
2. Full-dataset range distributions differ and create a potential confounder.
3. Restriction to 30–75 m produces comparable ranges.
4. Doppler-feature differences persist after range control.
5. Differences persist after aggregation by measurement session.
6. Bandwidth, entropy, near-zero energy, and high-velocity energy remain significant after Bonferroni correction.
7. Subtypes are heterogeneous, especially drone models D1, D4, and D5.

Bird sessions generally have broader, less concentrated spectra and greater high-velocity energy. Drone sessions generally have narrower spectra and greater near-zero concentration. Overlap supports a multivariate classifier rather than manual thresholds.

### Methodological Decisions

Subsequent experiments will:

- Exclude field-of-view edge samples.
- Merge D1–D6 into the drone class.
- Merge all bird categories into the bird class.
- Keep validation and test observations entirely real.
- Use macro-F1 and balanced accuracy as primary metrics.
- Report performance for every original subtype.
- Compare the official split with a session-independent split.
- Include full-range and range-controlled evaluation.
- Add synthetic samples only to training.

The complete range–Doppler tensor will be retained as CNN input because it preserves the analysed features plus additional spatial–spectral structure. Results from this radar dataset do not automatically establish generalisation to different sensors, frequencies, environments, or acquisition protocols.

The next notebook, **04_data_preprocessing.ipynb**, will construct reproducible model-ready tensors, labels, metadata, and limited-training-data subsets.